# FraudShield Advanced: Cost Optimization and Security via SDK

## Concept Review: Cost Optimization on SageMaker

SageMaker costs come from three primary sources: training compute, endpoint
hosting, and storage. Each has specific optimization levers:

| Cost Source | Optimization Strategy |
|---|---|
| **Training compute** | Managed Spot Training (up to 90% savings), instance right-sizing, stopping early when metrics plateau |
| **HPO compute** | Spot instances across all trials, limiting parallel jobs, using Bayesian optimization to converge faster |
| **Endpoint hosting** | Auto-scaling to match traffic, Serverless for bursty workloads, right-sized instances |
| **Storage** | S3 lifecycle policies, cleaning up old data capture files, archiving unused model artifacts |

This notebook covers the SDK patterns for Managed Spot Training, HPO with
Spot, auto-scaling policies, instance right-sizing via CloudWatch analysis,
and VPC/KMS security configuration.

---

*This notebook runs on ml.t3.medium in SageMaker Studio. Training jobs use
ml.m5.xlarge (Free Tier eligible).*

In [ ]:
%pip install -q sagemaker boto3 pandas

import sagemaker
import boto3
import json
import time
from datetime import datetime, timedelta
from sagemaker.xgboost import XGBoost
from sagemaker.tuner import (
    HyperparameterTuner,
    IntegerParameter,
    ContinuousParameter,
)

In [ ]:
sm_session = sagemaker.Session()
sm_client = sm_session.sagemaker_client
region = sm_session.boto_region_name
role = sagemaker.get_execution_role()
default_bucket = sm_session.default_bucket()

PREFIX = "fraudshield-cost-security"
TRAIN_S3 = f"s3://{default_bucket}/{PREFIX}/processed/train.csv"
VALIDATION_S3 = f"s3://{default_bucket}/{PREFIX}/processed/test.csv"
CHECKPOINT_S3 = f"s3://{default_bucket}/{PREFIX}/checkpoints"
OUTPUT_S3 = f"s3://{default_bucket}/{PREFIX}/output"

print(f"Region         : {region}")
print(f"Role           : {role.split('/')[-1]}")
print(f"Default bucket : {default_bucket}")

## Concept Review: Managed Spot Training

AWS Spot Instances are spare EC2 capacity available at up to 90% discount
compared to on-demand pricing. The trade-off: AWS can interrupt a Spot
Instance with two minutes' notice when demand for that capacity increases.

SageMaker Managed Spot Training integrates Spot Instances directly into
training jobs with built-in checkpoint management and automatic retry:

1. SageMaker requests Spot capacity instead of on-demand
2. If interrupted, SageMaker saves state to the checkpoint S3 location
3. When new capacity is available, training resumes from the last checkpoint
4. SageMaker tracks billable time vs. elapsed time separately

**Checkpointing is required** for Spot Training to be practical. Without it,
an interruption after hours of training loses all progress. Built-in
algorithms (XGBoost, K-Means) handle checkpointing automatically. Script
Mode training requires explicit checkpoint save/load logic in your code.

Three key parameters:

- `use_spot_instances=True` -- enables Spot pricing
- `max_wait` -- maximum total time including waits and interruptions
  (must exceed `max_run`)
- `checkpoint_s3_uri` -- S3 path for checkpoint storage

In [ ]:
spot_estimator = XGBoost(
    entry_point="train.py",
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    framework_version="1.5-1",
    output_path=OUTPUT_S3,
    use_spot_instances=True,
    max_wait=7200,
    max_run=3600,
    checkpoint_s3_uri=CHECKPOINT_S3,
    hyperparameters={
        "max_depth": 5,
        "eta": 0.2,
        "objective": "binary:logistic",
        "num_round": 100,
        "eval_metric": "auc",
        "scale_pos_weight": 10,
    },
)

print("Spot Estimator configured:")
print(f"  use_spot_instances : True")
print(f"  max_run            : 3600 seconds (1 hour)")
print(f"  max_wait           : 7200 seconds (2 hours)")
print(f"  checkpoint_s3_uri  : {CHECKPOINT_S3}")
print(f"  instance_type      : ml.m5.xlarge")

In [ ]:
training_script = '''
import argparse
import os
import json
import pickle
import xgboost as xgb
import pandas as pd

CHECKPOINT_DIR = "/opt/ml/checkpoints"


def load_checkpoint():
    checkpoint_path = os.path.join(CHECKPOINT_DIR, "xgb_checkpoint.pkl")
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "rb") as f:
            return pickle.load(f)
    return None


def save_checkpoint(model, epoch):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    checkpoint_path = os.path.join(CHECKPOINT_DIR, "xgb_checkpoint.pkl")
    with open(checkpoint_path, "wb") as f:
        pickle.dump({"model": model, "epoch": epoch}, f)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--max_depth", type=int, default=5)
    parser.add_argument("--eta", type=float, default=0.2)
    parser.add_argument("--num_round", type=int, default=100)
    parser.add_argument("--objective", type=str, default="binary:logistic")
    parser.add_argument("--eval_metric", type=str, default="auc")
    parser.add_argument("--scale_pos_weight", type=float, default=10)
    args, _ = parser.parse_known_args()

    train_path = os.path.join(os.environ["SM_CHANNEL_TRAIN"], "train.csv")
    train_df = pd.read_csv(train_path)
    dtrain = xgb.DMatrix(train_df.iloc[:, 1:], label=train_df.iloc[:, 0])

    checkpoint = load_checkpoint()
    start_epoch = 0
    xgb_model = None
    if checkpoint:
        xgb_model = checkpoint["model"]
        start_epoch = checkpoint["epoch"]
        print(f"Resuming from checkpoint at epoch {start_epoch}")

    params = {
        "max_depth": args.max_depth,
        "eta": args.eta,
        "objective": args.objective,
        "eval_metric": args.eval_metric,
        "scale_pos_weight": args.scale_pos_weight,
    }

    remaining = args.num_round - start_epoch
    model = xgb.train(
        params, dtrain, num_boost_round=remaining, xgb_model=xgb_model
    )
    save_checkpoint(model, args.num_round)

    model_dir = os.environ["SM_MODEL_DIR"]
    model.save_model(os.path.join(model_dir, "xgboost-model"))
    print("Training complete.")
'''

print("%%writefile train.py")
print(training_script)
print()
print("This training script demonstrates checkpoint save/load logic.")
print("On Spot interruption, SageMaker syncs /opt/ml/checkpoints to S3.")
print("On resume, the script detects the checkpoint and continues from")
print("where it left off instead of restarting from epoch 0.")

In [ ]:
spot_estimator.fit(
    inputs={"train": TRAIN_S3, "validation": VALIDATION_S3},
    wait=True,
    logs="All",
)

job_name = spot_estimator.latest_training_job.name
job_desc = sm_client.describe_training_job(TrainingJobName=job_name)

billable = job_desc.get("BillableTimeInSeconds", 0)
elapsed = job_desc.get("TrainingTimeInSeconds", 0)

print(f"\nTraining Job        : {job_name}")
print(f"Billable Seconds    : {billable}")
print(f"Elapsed Seconds     : {elapsed}")

spot_usage = job_desc.get("ManagedSpotTrainingUsage", {})
if spot_usage:
    savings = spot_usage.get("SavingsPercentage", 0)
    print(f"Spot Savings        : {savings}%")
else:
    if elapsed > 0:
        estimated_savings = max(0, (1 - billable / elapsed) * 100)
        print(f"Estimated Savings   : {estimated_savings:.1f}%")

## Concept Review: Spot Instances for HPO

HPO jobs launch dozens or hundreds of individual training trials to explore
the hyperparameter space. Combining HPO with Spot multiplies the savings
because every trial runs at a discount.

Spot interruptions interact differently with each HPO strategy:

- **Random Search:** No impact. Trials are independent; an interrupted trial
  is simply a failed data point.
- **Bayesian Optimization:** Minimal impact. The optimizer maintains
  uncertainty estimates and is robust to missing trials.
- **Hyperband:** Moderate impact. Successive halving depends on comparing all
  trials in a bracket; an interrupted trial may be unfairly eliminated.

Best practices: set generous `max_wait` (2-3x expected training time per
trial), keep parallel trials moderate (4-10) to improve capacity availability,
and always enable checkpointing.

In [ ]:
spot_hpo_estimator = XGBoost(
    entry_point="train.py",
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    framework_version="1.5-1",
    output_path=OUTPUT_S3,
    use_spot_instances=True,
    max_wait=7200,
    max_run=3600,
    checkpoint_s3_uri=CHECKPOINT_S3,
    hyperparameters={
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "scale_pos_weight": 10,
    },
)

tuner = HyperparameterTuner(
    estimator=spot_hpo_estimator,
    objective_metric_name="validation:auc",
    objective_type="Maximize",
    hyperparameter_ranges={
        "max_depth": IntegerParameter(3, 10),
        "eta": ContinuousParameter(0.01, 0.3),
        "num_round": IntegerParameter(50, 300),
    },
    max_jobs=10,
    max_parallel_jobs=3,
    strategy="Bayesian",
)

print("HyperparameterTuner configured with Spot:")
print(f"  max_jobs            : 10")
print(f"  max_parallel_jobs   : 3")
print(f"  strategy            : Bayesian")
print(f"  use_spot_instances  : True (inherited from base estimator)")
print(f"  objective_metric    : validation:auc (Maximize)")
print()
print("All 10 trials will use Spot pricing. At ~70% savings per trial,")
print("total HPO cost is roughly 30% of on-demand pricing.")

## Concept Review: Auto-scaling Policies

A real-time endpoint with a fixed instance count either wastes money during
low traffic or drops requests during spikes. SageMaker uses Application
Auto Scaling to dynamically adjust instance count.

The configuration has three components:

1. **Scalable target** -- registers the endpoint variant as a scalable
   resource with min and max instance counts
2. **Scaling policy** -- defines when and how much to scale based on metrics
3. **Cooldown periods** -- prevent rapid scale-out/scale-in flapping

**Target tracking** is the most common policy type. You specify a target
value for `SageMakerVariantInvocationsPerInstance` (average invocations per
instance per minute). SageMaker automatically adds instances when actual
invocations exceed the target and removes them when they drop below.

Cooldown best practices:

- **Scale-out cooldown:** Keep short (60-120 seconds) to respond quickly to
  traffic spikes
- **Scale-in cooldown:** Keep longer (300-600 seconds) to avoid removing
  instances during temporary traffic lulls

In [ ]:
aas_client = boto3.client("application-autoscaling", region_name=region)

ENDPOINT_NAME_FOR_SCALING = "fraudshield-production-ep"
VARIANT_NAME = "AllTraffic"

resource_id = f"endpoint/{ENDPOINT_NAME_FOR_SCALING}/variant/{VARIANT_NAME}"

aas_client.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=1,
    MaxCapacity=4,
)

aas_client.put_scaling_policy(
    PolicyName="FraudShieldTargetTracking",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 70.0,
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "SageMakerVariantInvocationsPerInstance",
        },
        "ScaleInCooldown": 300,
        "ScaleOutCooldown": 60,
    },
)

print(f"Auto-scaling configured for endpoint: {ENDPOINT_NAME_FOR_SCALING}")
print(f"  Min instances     : 1")
print(f"  Max instances     : 4")
print(f"  Target metric     : SageMakerVariantInvocationsPerInstance")
print(f"  Target value      : 70 invocations/instance/minute")
print(f"  Scale-in cooldown : 300 seconds")
print(f"  Scale-out cooldown: 60 seconds")

In [ ]:
policies = aas_client.describe_scaling_policies(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
)

for policy in policies.get("ScalingPolicies", []):
    print(f"Policy Name : {policy['PolicyName']}")
    print(f"Policy Type : {policy['PolicyType']}")
    print(f"Resource    : {policy['ResourceId']}")
    config = policy.get("TargetTrackingScalingPolicyConfiguration", {})
    if config:
        print(f"Target Value     : {config.get('TargetValue')}")
        print(f"Scale-in Cooldown: {config.get('ScaleInCooldown')} seconds")
        print(f"Scale-out Cooldown: {config.get('ScaleOutCooldown')} seconds")
        metric = config.get("PredefinedMetricSpecification", {})
        print(f"Metric Type      : {metric.get('PredefinedMetricType')}")
    print()

## Concept Review: Instance Right-sizing

Choosing the right instance type is one of the highest-impact cost decisions.
An oversized instance wastes money; an undersized one causes OOM errors or
high latency.

Right-sizing workflow:

1. Start with `ml.m5.xlarge` (Free Tier, 4 vCPUs, 16 GB RAM)
2. Monitor CloudWatch metrics during training: `CPUUtilization`,
   `MemoryUtilization`, `DiskUtilization`
3. Identify the bottleneck:
   - CPU > 90%, Memory < 50% -- CPU-bound, move to compute-optimized
     (`ml.c5.xlarge`)
   - Memory > 90% -- memory-bound, move to memory-optimized
     (`ml.r5.xlarge`)
   - Both < 30% -- oversized, move to a smaller instance
   - GPU needed -- deep learning only (`ml.g5.xlarge`)

Common mistake: using GPU instances for tabular ML (XGBoost, scikit-learn).
The built-in SageMaker containers use CPU, so GPU instances cost 5-10x more
with no speedup.

### Scaling Activity Monitoring

After configuring auto-scaling, monitor the scaling events to verify the
policy behaves as expected. Key metrics to watch in CloudWatch:

- `InvocationsPerInstance` -- should hover near the target value when scaling
  is working correctly
- `InstanceCount` -- the current number of active instances, showing scale-out
  and scale-in events over time
- `ModelLatency` -- if latency increases despite scaling, the bottleneck may
  be model inference time rather than instance count

In [ ]:
scaling_activities = aas_client.describe_scaling_activities(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MaxResults=5,
)

activities = scaling_activities.get("ScalingActivities", [])
if activities:
    print("Recent scaling activities:")
    for act in activities:
        print(f"  Time   : {act.get('StartTime', 'N/A')}")
        print(f"  Status : {act.get('StatusCode', 'N/A')}")
        print(f"  Cause  : {act.get('Cause', 'N/A')[:120]}")
        print()
else:
    print("No scaling activities recorded yet.")
    print("Activities appear after traffic triggers a scale-out or scale-in event.")

In [ ]:
cw_client = boto3.client("cloudwatch", region_name=region)

TRAINING_JOB_NAME = job_name if 'job_name' in dir() else "fraudshield-spot-training"

end_time = datetime.utcnow()
start_time = end_time - timedelta(hours=2)

metrics_to_check = ["CPUUtilization", "MemoryUtilization", "DiskUtilization"]

print(f"CloudWatch metrics for training job: {TRAINING_JOB_NAME}")
print(f"Time range: {start_time.isoformat()} to {end_time.isoformat()}")
print()

for metric_name in metrics_to_check:
    response = cw_client.get_metric_statistics(
        Namespace="/aws/sagemaker/TrainingJobs",
        MetricName=metric_name,
        Dimensions=[
            {"Name": "Host", "Value": f"{TRAINING_JOB_NAME}/algo-1"},
        ],
        StartTime=start_time,
        EndTime=end_time,
        Period=300,
        Statistics=["Average", "Maximum"],
    )

    datapoints = response.get("Datapoints", [])
    if datapoints:
        avg_values = [dp["Average"] for dp in datapoints]
        max_values = [dp["Maximum"] for dp in datapoints]
        overall_avg = sum(avg_values) / len(avg_values)
        overall_max = max(max_values)
        print(f"{metric_name}:")
        print(f"  Average: {overall_avg:.1f}%")
        print(f"  Peak:    {overall_max:.1f}%")
    else:
        print(f"{metric_name}: No data points found.")
    print()

In [ ]:
def recommend_instance(cpu_avg, mem_avg, current_type="ml.m5.xlarge"):
    """Produce a right-sizing recommendation from average utilization."""
    print(f"Current instance: {current_type}")
    print(f"CPU utilization (avg): {cpu_avg:.1f}%")
    print(f"Memory utilization (avg): {mem_avg:.1f}%")
    print()

    if cpu_avg > 90 and mem_avg < 50:
        print("Recommendation: CPU-bound workload.")
        print("  Move to compute-optimized: ml.c5.xlarge or ml.c5.2xlarge")
    elif mem_avg > 90:
        print("Recommendation: Memory-bound workload.")
        print("  Move to memory-optimized: ml.r5.xlarge or ml.r5.2xlarge")
    elif cpu_avg < 30 and mem_avg < 30:
        print("Recommendation: Instance is oversized.")
        print("  Move to smaller instance: ml.m5.large or ml.t3.xlarge")
    elif cpu_avg > 80 and mem_avg > 80:
        print("Recommendation: Both CPU and Memory are saturated.")
        print("  Scale up: ml.m5.2xlarge or ml.m5.4xlarge")
    else:
        print("Recommendation: Current instance is appropriately sized.")
        print(f"  Keep {current_type}.")


recommend_instance(cpu_avg=72.0, mem_avg=35.0)
print()
print("---")
print()
recommend_instance(cpu_avg=15.0, mem_avg=12.0)

## Concept Review: VPC and KMS Security

Production SageMaker deployments require three security layers:

**VPC isolation** -- SageMaker resources run in private subnets with no
internet access. S3 access is routed through a VPC Gateway Endpoint.
SageMaker API calls stay on the AWS backbone via PrivateLink Interface
Endpoints. This ensures no ML data or API calls traverse the public
internet.

**KMS encryption** -- Customer-managed KMS keys encrypt:

- Training job EBS volumes (`VolumeKmsKeyId` in `ResourceConfig`)
- Model artifacts and output data in S3 (SSE-KMS)
- Inter-container traffic in distributed training
  (`EnableInterContainerTrafficEncryption`)

**Network isolation** -- the strictest mode. When enabled, the training
container cannot make any outbound network calls. All code and dependencies
must be in the container image or passed through input channels.

The Estimator SDK parameters for VPC and KMS configuration:

- `subnets` -- list of private subnet IDs
- `security_group_ids` -- list of security group IDs
- `volume_kms_key` -- KMS key ARN for EBS encryption
- `encrypt_inter_container_traffic` -- boolean for distributed training

In [ ]:
VPC_SUBNETS = ["subnet-0abc123def456789a", "subnet-0def456789abc123b"]
SECURITY_GROUPS = ["sg-0abc123def456789a"]
KMS_KEY_ARN = "arn:aws:kms:us-east-1:123456789012:key/abcd1234-ef56-7890-gh12-ijklmnopqrst"

secure_estimator = XGBoost(
    entry_point="train.py",
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    framework_version="1.5-1",
    output_path=OUTPUT_S3,
    subnets=VPC_SUBNETS,
    security_group_ids=SECURITY_GROUPS,
    volume_kms_key=KMS_KEY_ARN,
    encrypt_inter_container_traffic=True,
    hyperparameters={
        "max_depth": 5,
        "eta": 0.2,
        "objective": "binary:logistic",
        "num_round": 100,
        "eval_metric": "auc",
    },
)

print("Secure Estimator configuration (not executed, pattern only):")
print(f"  VPC subnets              : {VPC_SUBNETS}")
print(f"  Security groups          : {SECURITY_GROUPS}")
print(f"  Volume KMS key           : {KMS_KEY_ARN[:50]}...")
print(f"  Inter-container encrypt  : True")
print()
print("This configuration ensures:")
print("  - Training runs in private subnets (no internet access)")
print("  - EBS volumes are encrypted with a customer-managed KMS key")
print("  - Data exchanged between instances is encrypted in transit")
print("  - S3 access routes through a VPC Gateway Endpoint (must exist in VPC)")

## Key Takeaways

**Cost optimization levers:**

1. **Managed Spot Training** -- up to 90% savings on training compute;
   requires checkpointing for interruption recovery
2. **Spot + HPO** -- multiply Spot savings across all tuning trials;
   Bayesian and Random strategies are robust to interruptions
3. **Auto-scaling** -- target tracking on `InvocationsPerInstance` to
   scale endpoints dynamically; short scale-out cooldown, long scale-in
4. **Instance right-sizing** -- use CloudWatch CPU/Memory metrics to
   identify the bottleneck and select the optimal instance family

**Security configuration:**

1. **VPC isolation** -- private subnets, VPC Endpoints for S3 and
   SageMaker APIs, no internet access
2. **KMS encryption** -- customer-managed keys for EBS volumes, S3
   artifacts, and inter-container traffic
3. **Network isolation** -- strictest mode, prevents all outbound
   network calls from training containers

These patterns apply to every SageMaker workload. Start with
`ml.m5.xlarge` (Free Tier), enable Spot for non-time-critical training,
and configure auto-scaling for any production endpoint.